# Chương 7 — Hệ đa tác tử: học tăng cường (Mèo đuổi chuột) và tác tử LLM tranh luận (MedMCQA)

In [1]:
# MAS-00 · Thiết lập đường dẫn: mã chương dùng chung gói của tiểu luận (src/dm) và mã kế thừa của nhóm (src/tl01, tl03)
import os, sys, json, time, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""
from pathlib import Path
warnings.filterwarnings("ignore")
TLF = Path.cwd().parent
for p in ["src", "src/tl01", "src/tl03"]:
    sys.path.insert(0, str(TLF / p))
import numpy as np, pandas as pd
np.set_printoptions(precision=4, suppress=True)
print("Thư mục tiểu luận:", TLF)

Thư mục tiểu luận: /home/ngtukien/Projects/MultiAgent/tieuluan_final


Notebook tổng hợp hai bài thực nghiệm của Chương 7: (1) hệ đa tác tử học tăng cường trên lưới 8×8 (một mèo, hai mèo hợp tác IQL, mèo–chuột học đối kháng) và (2) hệ đa tác tử LLM trả lời 150 câu hỏi trắc nghiệm đái tháo đường MedMCQA.

In [2]:
# MAS-01 · Kết quả ba thí nghiệm học tăng cường Mèo đuổi chuột (đơn tác tử, hợp tác 2 mèo IQL, đối kháng)
cm = json.loads((TLF / "outputs/metrics/ch7_catmouse.json").read_text())
print("Thí nghiệm A (1 mèo vs chuột luật):")
display(pd.DataFrame(cm["A"]["eval"]).round(3))
print("Thí nghiệm B (2 mèo hợp tác):", {k: cm["B"][k] for k in ("n_states", "rule_rule", "iql")})
print("Thí nghiệm C (Ma trận đối kháng 3 mèo × 2 chuột):")
pd.DataFrame(cm["C"]["table"]).round(3)

Thí nghiệm A (1 mèo vs chuột luật):


,agent,catch_rate,avg_steps,avg_reward
0,Random Agent,0.023,98.530,-154.293
1,Rule-Based Chaser,0.243,82.527,-189.450
2,Q-Learning Agent,1.000,15.263,83.837


Thí nghiệm B (2 mèo hợp tác): {'n_states': 157464, 'rule_rule': {'catch_rate': 0.52, 'avg_steps': 61.86333333333334}, 'iql': {'catch_rate': 0.9966666666666667, 'avg_steps': 22.133333333333333}}
Thí nghiệm C (Ma trận đối kháng 3 mèo × 2 chuột):


,cat,mouse,catch_rate,avg_steps
0,Q (học với chuột luật),chuột luật (chạy trốn),1.000,15.263
1,Q (học với chuột luật),chuột Q (học đối kháng),0.037,97.317
2,Q (học đối kháng),chuột luật (chạy trốn),1.000,16.337
3,Q (học đối kháng),chuột Q (học đối kháng),0.747,42.290
4,Rule-Based,chuột luật (chạy trốn),0.243,82.527
5,Rule-Based,chuột Q (học đối kháng),0.047,96.250


In [3]:
# MAS-02 · Kết quả 4 hệ tác tử LLM trên 150 câu hỏi trắc nghiệm đái tháo đường (MedMCQA + MedQuAD)
med = json.loads((TLF / "outputs/metrics/ch7_medqa.json").read_text())["systems"]
pd.DataFrame(med).T[["n", "accuracy", "calls", "tokens_out", "seconds", "errors"]].round(3)

,n,accuracy,calls,tokens_out,seconds,errors
direct,150,0.64,1.0,4.0,0.216027,{'knowledge': 54}
cot,150,0.573333,1.0,98.533333,4.955313,"{'knowledge': 63, 'no_answer': 1}"
rag,150,0.54,1.0,31.733333,2.042393,{'knowledge': 69}
multi,150,0.56,2.84,207.893333,8.92264,"{'knowledge': 43, 'coordination': 23}"


In [4]:
# MAS-03 · Phân tích cổng đồng thuận (Verifier) và cận trên Oracle của hệ đa tác tử trên 150 câu MedMCQA
from dm.ch7_medqa import to_letter
rows = {s: {r["id"]: r for r in map(json.loads, (TLF / f"outputs/ch7/rows_{s}.jsonl").read_text().splitlines())}
        for s in ("direct", "cot", "rag", "multi")}
agree, dis, oracle = [], [], 0
for i in range(150):
    m = rows["multi"][i]
    disagreed = any(s["role"] == "Coordinator" for s in m["steps"])
    (dis if disagreed else agree).append(m["correct"])
    picks = [rows["cot"][i]["pred"], rows["rag"][i]["pred"]] + [to_letter(s["text"]) for s in m["steps"] if s["role"] == "Critic"]
    oracle += (m["gold"] in picks)
print(f"Đồng thuận vòng 1: {len(agree)}/150 câu ({len(agree)/150:.1%}) -> Accuracy = {np.mean(agree):.3f}")
print(f"Bất đồng (cần Critic + Vote): {len(dis)}/150 câu ({len(dis)/150:.1%}) -> Accuracy = {np.mean(dis):.3f}")
print(f"Cận trên Oracle (ít nhất 1 trong CoT/RAG/Critic đúng): {oracle}/150 ({oracle/150:.3f})")

Đồng thuận vòng 1: 87/150 câu (58.0%) -> Accuracy = 0.701
Bất đồng (cần Critic + Vote): 63/150 câu (42.0%) -> Accuracy = 0.365
Cận trên Oracle (ít nhất 1 trong CoT/RAG/Critic đúng): 107/150 (0.713)


In [5]:
# MAS-04 · Xem vết tranh luận thật ở câu #18 (RAG cứu CoT) và câu #101 (CoT cứu RAG)
from dm.ch7_medqa import load_questions
qs = {q["id"]: q for q in load_questions(150)}
for qid in (18, 101):
    q, m = qs[qid], rows["multi"][qid]
    print(f"=== Câu #{qid} ({q['subject']}) | Đáp án đúng: {q['gold']} | Dự đoán Multi: {m['pred']} ===")
    print("Câu hỏi:", q["question"])
    print("Phương án:", {chr(65+k): v for k, v in enumerate(q["options"])})
    for s in m["steps"]:
        print(f"  [{s['role']:12s}]: {s['text'][:160].replace(chr(10), ' ')}")

=== Câu #18 (Pediatrics) | Đáp án đúng: D | Dự đoán Multi: D ===
Câu hỏi: Which of the following is a risk factor for developing Type 1 Diabetes mellitus?
Phương án: {'A': 'Male Gender', 'B': 'Old age', 'C': 'Gestational diabetes', 'D': 'HLA DR3'}
  [Solver-CoT  ]: Type 1 Diabetes mellitus is an autoimmune disease characterized by the destruction of insulin-producing beta cells in the pancreas. Risk factors include genetic
  [Tool:search ]: [1] Causes of Diabetes — What causes Causes of Diabetes ? (BM25 41.3); [2] Causes of Diabetes — What causes Causes of Diabetes ? (BM25 41.3); [3] Causes of Diab
  [Solver-RAG  ]: Answer: D. HLA DR3
  [Critic      ]: Colleague 1 is correct.  The reference passage clearly states that "Type 1 diabetes is caused by a lack of insulin due to the destruction of insulin-producing b
  [Solver-CoT#2]: Step 1: Identify the type of diabetes being discussed. Step 2: Recall that Type 1 Diabetes Mellitus (T1DM) is an autoimmune disorder characterized by insufficie